In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install gradio

In [ ]:
!pip install -U langchain-community
!pip install faiss-cpu
!pip install rank_bm25
!pip install transformers==4.44.2
!pip install -U FlagEmbedding

In [ ]:
import os
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.load import dumps, loads
from collections import defaultdictи
from openai import OpenAI, AzureOpenAI
from FlagEmbedding import FlagReranker
import gradio as gr

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

nebius_key=os.getenv("NEBIUS_API_KEY")
nebius_base_url=os.getenv("NEBIUS_BASE_URL")

In [ ]:
base_url=os.getenv("NEBIUS_BASE_URL"),
nebius_api_key=os.getenv("NEBIUS_API_KEY"),
mistral_model = "mistralai/Mistral-Nemo-Instruct-2407"

In [ ]:
embedding_folder = "/content/drive/MyDrive/Colab Notebooks/embedded_chunks"
faiss_index_path = "/content/drive/MyDrive/Colab Notebooks/faiss"

In [ ]:
def load_data():
    meta_path = os.path.join(embedding_folder, "metadata.json")
    texts_path = os.path.join(embedding_folder, "texts.json")
    embeddings_path = os.path.join(embedding_folder, "embeddings.npy")
    embeddings = np.load(embeddings_path)
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    with open(texts_path, "r", encoding="utf-8") as f:
        texts = json.load(f)
    return metadata, texts

In [ ]:
class DeepVKEmbeddings(Embeddings):
    def __init__(self, model_name="deepvk/USER-base", device="cpu", batch_size=16):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.device = device
        self.batch_size = batch_size
        self.model.to(device)
        self.model.eval()

    @staticmethod
    def mean_pooling(model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        summed = torch.sum(token_embeddings * mask, dim=1)
        count = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / count

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        all_emb = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i : i + self.batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=512)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                out = self.model(**inputs)
                emb = self.mean_pooling(out, inputs["attention_mask"]).cpu().numpy()
                all_emb.extend(emb)
        return all_emb

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]


device = "cuda" if torch.cuda.is_available() else "cpu"
metadata, texts = load_data()
emb_model = DeepVKEmbeddings(device=device)

In [ ]:
faiss_vs = FAISS.load_local(
    faiss_index_path,
    embeddings=emb_model,
    allow_dangerous_deserialization=True
)
faiss_ret = faiss_vs.as_retriever(search_kwargs={"k": 15})
docs = [Document(page_content=t, metadata={**m, "document_id": i}) for i, (t, m) in enumerate(zip(texts, metadata))]
bm25_ret = BM25Retriever.from_documents(docs)
bm25_ret.k = 15

In [ ]:
def reciprocal_rank_fusion(results: list[list], k: int = 60):
    from collections import defaultdict
    fused = defaultdict(float)
    for docs in results:
        for rank, d in enumerate(docs):
            ds = dumps(d)
            fused[ds] += 1 / (rank + k)
    return [(loads(ds), score) for ds, score in sorted(fused.items(), key=lambda x: x[1], reverse=True)]

er = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)
ensemble_ret = EnsembleRetriever(retrievers=[faiss_ret, bm25_ret])
client_nb = OpenAI(
        base_url=os.getenv("NEBIUS_BASE_URL"), 
        nebius_api_key=os.getenv("NEBIUS_API_KEY"),
)

In [ ]:
def generate_answer(question: str, top_k: int = 10) -> tuple[str, list, list]:
    f_docs = faiss_ret.get_relevant_documents(question)
    b_docs = bm25_ret.get_relevant_documents(question)
    rrf = reciprocal_rank_fusion([f_docs, b_docs])
    docs, _ = zip(*rrf)

    pairs = [[question, d.page_content] for d in docs]
    scores = er.compute_score(pairs, normalize=True)
    scored = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    seen = set(); unique = []
    for d, sc in scored:
        key = (d.metadata.get("chunk_id"), d.metadata.get("file_name"))
        if key not in seen:
            seen.add(key); unique.append((d, sc))
        if len(unique) == top_k: break
    reranked_docs, reranked_scores = zip(*unique) if unique else ([], [])

    context = "\n\n".join(d.page_content for d in reranked_docs)

    prompt = f"""
    Ты — медицинский консультант. Отвечай строго на русском языке.
    Используй только текст инструкции для ответа.
    Дай чёткий и краткий ответ на вопрос пациента.
    Если в инструкции нет ответа, то посоветуй обратиться к врачу.
    
    Инструкция:\n{context}
    Вопрос пациента:\n{question}
    """.strip()

    try:
        resp = client_nb.chat.completions.create(
            model=mistral_model,
            messages=[
                {"role": "system", "content":
                 "Ты — медицинский консультант и отвечаешь на вопросы пациентов."
                "Давай ответы только на русском языке на основе текста инструкций."
                },
                {"role": "user", "content": prompt}
            ],
            max_tokens=800,
            temperature=0.2
        )
        answer = resp.choices[0].message.content.strip()
    except Exception as e:
        answer = "Ошибка при генерации: " + str(e)

    return answer, list(reranked_docs), list(reranked_scores)

**RAG-пайплайн для чат-бота**

In [ ]:
def rag_pipeline(user_message: str, history: list[tuple]) -> tuple[list[tuple], list[tuple]]:
    answer, docs, scores = generate_answer(user_message)
    history = history + [(user_message, answer)]
    return history, history

**Gradio UI**

In [ ]:
custom_css = """

label:has-text("Textbox") {
    display: none;
}

/* рамка поля ввода */
textarea, .gr-textbox {
    border: 2px solid #DCDCDC !important;
}

/* placeholder — яркий, жирный */
textarea::placeholder {
    color: #808080;
    font-weight: 600;
    font-size: 16px;
}

/* стиль текста в сообщениях */
.gr-message {
    font-size: 18px;
    font-weight: 500;
    color: #1f1f1f;
}
"""

with gr.Blocks(css=custom_css) as demo:
    chatbot = gr.Chatbot()
    state = gr.State([])
    txt = gr.Textbox(placeholder="Введите ваш вопрос…", show_label=False)
    txt.submit(rag_pipeline, [txt, state], [chatbot, state])
    txt.submit(lambda: "", None, txt)

demo.launch(share=True)

Пример работы

<img src="chatbot_example_query.png" style="width:100%;" />